In [1]:
from sklearn.metrics import classification_report
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from swin_transformer import swin_t


In [2]:
import os
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import torch.optim as optim
from sklearn.metrics import accuracy_score

class CustomDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.class_labels = ['glioma', 'meningioma', 'notumor', 'pituitary']
        self.images, self.labels = self.load_data()

    def load_data(self):
        images = []
        labels = []

        for label in self.class_labels:
            label_directory = os.path.join(self.data_dir, label)
            label_idx = self.class_labels.index(label)

            for filename in os.listdir(label_directory):
                img_path = os.path.join(label_directory, filename)
                images.append(img_path)
                labels.append(label_idx)

        return images, labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]

        # Load image
        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label

data_dir = r"Dataset_4C"

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

custom_dataset = CustomDataset(data_dir, transform=transform)

train_size = int(0.9 * len(custom_dataset))
test_size = len(custom_dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(custom_dataset, [train_size, test_size])

batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)


print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of testing samples: {len(test_dataset)}")

Number of training samples: 6320
Number of testing samples: 703


In [3]:
model_path = "SWIN-model-4C-100.pth"
model = swin_t()
model.load_state_dict(torch.load(model_path))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

y_true = []
y_pred = []
for inputs, labels in test_loader:
    # inputs = inputs.unsqueeze(1)
    inputs, labels = inputs.to(device), labels.to(device)
    outputs = model(inputs)
    _, predicted = torch.max(outputs, 1)
    y_true.extend(labels.cpu().numpy())
    y_pred.extend(predicted.cpu().numpy())

# Define class names
class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']

# Calculate precision, recall, and F1 score
print(classification_report(y_true, y_pred, target_names=class_names))


              precision    recall  f1-score   support

      glioma       1.00      0.99      1.00       166
  meningioma       0.99      0.99      0.99       163
     notumor       0.99      0.99      0.99       197
   pituitary       0.99      1.00      1.00       177

    accuracy                           1.00       703
   macro avg       1.00      1.00      1.00       703
weighted avg       1.00      1.00      1.00       703

